### DoubleDQN vs Random v1
#### Smaller Network, gamma: 0.5 (not squared), lr: 5e-5

In [5]:
import torch
from collections import Counter
from domain.configs import MAX_STEPS_PER_EPISODE
from environment.grenight_environment import GrenightEnvironment
from agents.double_dqn_vs_random.agent import Agent

In [14]:
def play_game(env_arg: GrenightEnvironment,
              agent_arg: Agent) -> tuple[str, dict, int]:

    state = env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        if acting_player_is_white:
            action = agent_arg.select_action(state, env_arg.action_mask(), 0.0)
        else:
            action = env_arg.sample()
        state, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", info, move_count
    if reward == 0.0:
        return "draw", info, move_count

    winner_is_white = acting_player_is_white if reward == 1.0 else not acting_player_is_white

    return ("white_win", info, move_count) if winner_is_white else ("black_win", info, move_count)

In [19]:
device = "cpu"

for ep in (10_000, 20_000, 30_000):
    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_vs_random/checkpoints/ep{ep}.pt", map_location=device, weights_only=False)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for _ in range(1000):
        outcome, game_info, move_count = play_game(env, agent)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1

    print(f"STATS OUT FROM: {1000} GAMES AT: {ep} CHECKPOINT\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 GAMES AT: 10000 CHECKPOINT
Outcomes: Counter({'white_win': 571, 'draw': 361, 'black_win': 68})
Draw reasons: Counter({'insufficient_material': 151, 'stalemate': 108, 'threefold_repetition': 76, 'max_steps_without_progress': 26})

STATS OUT FROM: 1000 GAMES AT: 20000 CHECKPOINT
Outcomes: Counter({'white_win': 689, 'draw': 258, 'black_win': 53})
Draw reasons: Counter({'insufficient_material': 94, 'stalemate': 92, 'threefold_repetition': 50, 'max_steps_without_progress': 22})

STATS OUT FROM: 1000 GAMES AT: 30000 CHECKPOINT
Outcomes: Counter({'white_win': 722, 'draw': 237, 'black_win': 41})
Draw reasons: Counter({'insufficient_material': 107, 'threefold_repetition': 65, 'stalemate': 48, 'max_steps_without_progress': 17})

